# Convert data from SpatialData into InSituPy

## Setup and Imports

In [1]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

In [2]:
# Import spatialdata first to prevent potential import conflicts
# See: https://github.com/scverse/spatialdata/pull/570
import spatialdata

c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\site-packages\spatialdata\_core\query\relational_query.py:530: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap 

In [3]:
from pathlib import Path

from insitupy import InSituData, CACHE
from insitupy.spatialdata import convert_from_spatialdata

c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load SpatialData

First, let's load a SpatialData object. We'll use the `spatialdata_io` package to read data from common spatial transcriptomics platforms like Xenium or MERSCOPE.

In this tutorial, we'll use a Xenium dataset as an example. If you did not download the demo datasets already, checkout the [demo dataset tutorial](../01_demo_analysis/00_InSituPy_demo_datasets.ipynb) to learn how to do so.

In [4]:
from spatialdata_io import xenium

In [5]:
# Path to your Xenium output folder
datapath = CACHE / "demo_datasets/xenium_hbreastcancer/output-XETG00000__slide_id__hbreastcancer"

In [6]:
# Load the Xenium data as a SpatialData object
sdata = xenium(datapath)

C:\Users\ge37voy\AppData\Local\Temp\ipykernel_21024\3619369563.py:2: DeprecationWarning: The default value of `cells_as_circles` will change to `False` in the next release. Please pass `True` explicitly to maintain the current behavior.
  sdata = xenium(datapath)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [7]:
# Display the SpatialData object to see available elements
sdata

SpatialData object
├── Images
│     ├── 'morphology_focus': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
│     └── 'morphology_mip': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
│     └── 'nucleus_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 8) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (167780, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (167780, 313)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), morphology_mip

To better understand the `SpatialData` structure, please look into the [SpatialData documentation](https://spatialdata.scverse.org/en/latest/index.html).

## Convert to InSituData

The `convert_from_spatialdata()` function converts a SpatialData object into an InSituData object. You need to specify which elements from the SpatialData object should be mapped to InSituPy's data structure.

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `sdata` | The SpatialData object to convert |
| `image_data` | Dictionary mapping image names to `(sdata_key, pixel_size)` tuples |
| `cells_key` | Key for cell shapes in SpatialData (e.g., `"cell_circles"`) |
| `table_key` | Key for the expression table (default: `"table"`) |
| `cell_boundaries_data` | Tuple of `(label_key, pixel_size)` for cell segmentation masks |
| `nucleus_boundaries_data` | Tuple of `(label_key, pixel_size)` for nucleus segmentation masks |
| `transcripts_key` | Key for transcript points (default: `"transcripts"`) |
| `slide_id` | Identifier for the slide |
| `sample_id` | Identifier for the sample |
| `method_name` | Name of the spatial method (e.g., `"Xenium"`) |

In [9]:
# Define the pixel size (in micrometers per pixel)
# For Xenium, this is typically 0.2125 µm/pixel
pixel_size = 0.2125

In [41]:
sdata["morphology_mip"].scale0['image'].data.squeeze(0)

dask.array<getitem, shape=(25778, 35416), dtype=uint16, chunksize=(4096, 4096), chunktype=numpy.ndarray>

In [35]:
len(sdata["morphology_mip"].scale0['image']['c'].data)

1

In [52]:
# Convert SpatialData to InSituData
xd = convert_from_spatialdata(
    sdata=sdata,
    # Map images: {new_name: (spatialdata_key, pixel_size)}
    image_data={
        "nuclei": ("morphology_mip", pixel_size, True),
        "mip": ("morphology_focus", pixel_size)
    },
    # Cell data configuration
    cells_key="cell_circles",
    table_key="table",
    # Boundary masks
    cell_boundaries_data=("cell_labels", pixel_size),
    nucleus_boundaries_data=("nucleus_labels", pixel_size),
    # Transcripts
    transcripts_key="transcripts",
    # Metadata
    slide_id="slide_demo",
    sample_id="sample_demo",
    method_name="Xenium"
)

INFO: Using 'global' coordinate system for pixel size extraction.


Adding images...
Adding cell data...


Adding transcripts...


In [54]:
# Display the converted InSituData object
xd

InSituData
Method:		Xenium
Slide ID:	slide_demo
Sample ID:	sample_demo
Path:		None

    ➤ images
       'nuclei':   (25778, 35416)
       'mip':      (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape Delayed('int-2de95c0c-a261-4a83-af79-9d61fab272f7') x 8

## Explore the Converted Data

Let's verify that all data modalities were correctly converted.

In [55]:
# Check available images
xd.images

'nuclei':   (25778, 35416)
'mip':      (25778, 35416)

In [57]:
# Check cell data and spatial coordinates
xd.cells.table.obsm['spatial']

array([[ 847.25991211,  326.19136505],
       [ 826.34199524,  328.03182983],
       [ 848.76691895,  331.74318695],
       ...,
       [7470.15942383, 5119.13205566],
       [7477.73720703, 5128.71281738],
       [7489.3765625 , 5123.19777832]], shape=(167780, 2))

In [58]:
# Check cell boundaries
xd.cells.boundaries

BoundariesData object with 2 entries:
    cells
    nuclei

In [59]:
# Check transcripts
xd.transcripts

,x_location,y_location,z_location,feature_name,cell_id,overlaps_nucleus,transcript_id,qv
npartitions=8,,,,,,,,
,float32,float32,float32,string,int32,uint8,uint64,float32
,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...


## Saving the Converted Data

Once converted, you can save the InSituData object to disk in InSituPy's native format for efficient storage and future use.

In [60]:
# Define output path
outpath = CACHE / "out/from_spatialdata_demo"

In [61]:
# Save the InSituData object
xd.saveas(outpath, overwrite=True)
print(f"Saved to: {outpath}")

Saving data to C:\Users\ge37voy\.cache\InSituPy\out\from_spatialdata_demo


... storing 'feature_types' as categorical
... storing 'genome' as categorical
c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\site-packages\zarr\core\dtype\npy\string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


Saved.
Saved to: C:\Users\ge37voy\.cache\InSituPy\out\from_spatialdata_demo


## Loading Saved Data

The saved data can be loaded back using InSituPy's standard reading functions.

In [62]:
# Load the saved data
xd_loaded = InSituData.read(outpath)
xd_loaded.load_all()

In [63]:
# Display the loaded data
xd_loaded

InSituData
Method:		Xenium
Slide ID:	slide_demo
Sample ID:	sample_demo
Path:		C:\Users\ge37voy\.cache\InSituPy\out\from_spatialdata_demo

    ➤ images
       'mip':      (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape Delayed('int-c942b548-cc6a-4bfb-9707-ad1e0bc77cee') x 8

In [65]:
# Visualize the data
xd_loaded.show()

## Alternative: Using the Xenium Reader with SpatialData Backend

InSituPy also provides a convenient way to read Xenium data directly using SpatialData as the backend. This combines the loading and conversion into a single step.

In [6]:
from insitupy.io import read_xenium

In [7]:
# Read Xenium data using SpatialData backend
xd_direct = read_xenium(datapath, backend="spatialdata")

INFO: Reading Xenium data with spatialdata-io backend...
c:\Users\ge37voy\AppData\Local\miniconda3\envs\ispsd\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
INFO: Using 'global' coordinate system for pixel size extraction.


Adding images...
Adding cell data...


Adding transcripts...


In [8]:
# Display the result
xd_direct

InSituData
Method:		Xenium
Slide ID:	slide_id
Sample ID:	sample_id
Path:		None

    ➤ images
       'nuclei':   (25778, 35416)
       'mip':      (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape Delayed('int-b828b4d8-a509-4bc9-b5a0-b0152142dcc3') x 8